In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np


file_path = r'' # path to the Flair.nii file
mask_path = r'' # path to the LesionSeg file

def visualize_mri(mri_path, mask_path):
    
    mri_img = nib.load(mri_path).get_fdata()
    mask_img = nib.load(mask_path).get_fdata()

  
    slice_idx = mri_img.shape[2] // 2

    plt.figure(figsize=(12, 6))

    
    plt.subplot(1, 2, 1)
    plt.imshow(mri_img[:, :, slice_idx].T, cmap='gray', origin='lower')
    plt.title(f'FLAIR MRI (Slice {slice_idx})')
    plt.axis('off')

   
    plt.subplot(1, 2, 2)
    plt.imshow(mri_img[:, :, slice_idx].T, cmap='gray', origin='lower')
    
    masked_data = np.ma.masked_where(mask_img[:, :, slice_idx] == 0, mask_img[:, :, slice_idx])
    plt.imshow(masked_data.T, cmap='autumn', origin='lower', alpha=0.6)
    plt.title('Expert Lesion Mask Overlay')
    plt.axis('off')

    plt.tight_layout()
    plt.show()


visualize_mri(file_path, mask_path)

In [ ]:
import os

base_dir = r"" # path to the data

images = []
labels = []

for root, dirs, files in os.walk(base_dir):
   
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue
    
    patient_num = folder_name.split("-")[1]  # "7"
    
    flair_file = f"{patient_num}-Flair.nii"
    label_file = f"{patient_num}-LesionSeg-Flair.nii"
    
    flair_path = os.path.join(root, flair_file)
    label_path = os.path.join(root, label_file)
    
    if os.path.exists(flair_path) and os.path.exists(label_path):
        images.append(flair_path)
        labels.append(label_path)
    else:
        print(f"⚠ Missing file in {folder_name}: "
              f"Flair={os.path.exists(flair_path)}, Label={os.path.exists(label_path)}")


images.sort(key=lambda x: int(os.path.basename(os.path.dirname(x)).split("-")[1]))
labels.sort(key=lambda x: int(os.path.basename(os.path.dirname(x)).split("-")[1]))

print(f"--- Search Results ---")
print(f"Total Flair Images:  {len(images)}")
print(f"Total Mask Labels:   {len(labels)}")

print(f"\n--- First 3 Pairs ---")
for img, lbl in zip(images[:3], labels[:3]):
    print(f"  Image : {img}")
    print(f"  Label : {lbl}")
    print()


data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(images, labels)]
print(f"data_dicts ready: {len(data_dicts)} entries")

In [ ]:
import os
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    Orientationd, ScaleIntensityRanged, CropForegroundd,
    RandCropByPosNegLabeld, ToTensord
)
from monai.data import Dataset, DataLoader


base_dir = r""

images, labels = [], []

for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue

    patient_num = folder_name.split("-")[1]
    flair_path = os.path.join(root, f"{patient_num}-Flair.nii")
    label_path = os.path.join(root, f"{patient_num}-LesionSeg-Flair.nii")

    if os.path.exists(flair_path) and os.path.exists(label_path):
        images.append(flair_path)
        labels.append(label_path)

images.sort(key=lambda x: int(os.path.basename(os.path.dirname(x)).split("-")[1]))
labels.sort(key=lambda x: int(os.path.basename(os.path.dirname(x)).split("-")[1]))

data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(images, labels)]


train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=1000, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=1, neg=1, num_samples=4,
        image_key="image",
        image_threshold=0,
    ),
    ToTensord(keys=["image", "label"]),
])


train_ds = Dataset(data=data_dicts, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=4)

print(f"Dataset ready with {len(train_ds)} volumes.")

In [ ]:

import os
import torch
from tqdm.notebook import tqdm
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    Orientationd, ScaleIntensityRanged, CropForegroundd,
    RandCropByPosNegLabeld, ToTensord, RandFlipd, RandRotate90d,
    AsDiscrete, SpatialPadd, DivisiblePadd   
)
from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.utils import set_determinism

In [ ]:
set_determinism(seed=42)

base_dir = r""

images, labels = [], []

for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue
    patient_num = folder_name.split("-")[1]
    flair_path = os.path.join(root, f"{patient_num}-Flair.nii")
    label_path = os.path.join(root, f"{patient_num}-LesionSeg-Flair.nii")
    if os.path.exists(flair_path) and os.path.exists(label_path):
        images.append(flair_path)
        labels.append(label_path)

images.sort(key=lambda x: int(os.path.basename(os.path.dirname(x)).split("-")[1]))
labels.sort(key=lambda x: int(os.path.basename(os.path.dirname(x)).split("-")[1]))

data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(images, labels)]

split      = int(0.8 * len(data_dicts))   
train_files = data_dicts[:split]
val_files   = data_dicts[split:]

print(f"Train: {len(train_files)} | Val: {len(val_files)}")

In [ ]:

from monai.transforms import SpatialPadd


CROP_SIZE = (96, 96, 32)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=1000, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    SpatialPadd(keys=["image", "label"], spatial_size=(-1, -1, 32)),
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label",
        spatial_size=CROP_SIZE,
        pos=1, neg=1, num_samples=2,
        image_key="image", image_threshold=0,
        allow_smaller=False,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    ToTensord(keys=["image", "label"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=1000, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    
    SpatialPadd(keys=["image", "label"], spatial_size=(
        -1, -1, 32  
    )),
 
    DivisiblePadd(keys=["image", "label"], k=32),
    ToTensord(keys=["image", "label"]),
])

print(f"Crop size: {CROP_SIZE}")
print("Transforms defined.")

In [ ]:
train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds   = Dataset(data=val_files,   transform=val_transforms)


train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
import os
import requests
import monai
from monai.inferers import SlidingWindowInferer
from monai.losses import DiceFocalLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

monai_version = tuple(int(x) for x in monai.__version__.split(".")[:2])

model = SwinUNETR(
    in_channels=1,
    out_channels=2,
    feature_size=24,
    use_checkpoint=True,
).to(device)


weights_path = "swin_unetr_pretrained.pt"

if not os.path.exists(weights_path):
    print("Downloading pretrained weights (~20MB)...")
    url = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/model_swinvit.pt"
    r = requests.get(url, stream=True)
    with open(weights_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")

try:
    weights = torch.load(weights_path, map_location=device)
    model.load_from(weights)
    print("✓ Pretrained weights loaded into SwinViT backbone.")
except Exception as e:
    print(f"⚠ Could not load pretrained weights: {e}")
    print("  Continuing without pretrained weights.")


loss_fn = DiceFocalLoss(
    to_onehot_y=True,
    softmax=True,
    gamma=2.0,          
    lambda_dice=0.5,
    lambda_focal=0.5,
)


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",       
    factor=0.5,       
    patience=10,      
    min_lr=1e-7,      
    verbose=True,     
)


inferer = SlidingWindowInferer(
    roi_size=(96, 96, 32),
    sw_batch_size=1,      
    overlap=0.25,
    mode="gaussian",
)


dice_metric = DiceMetric(include_background=False, reduction="mean")
post_pred   = AsDiscrete(argmax=True, to_onehot=2)
post_label  = AsDiscrete(to_onehot=2)

print(f"MONAI version      : {monai.__version__}")
print(f"Model ready.")
print(f"Trainable params   : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


RESUME_FROM = "best_swin_unetr.pth"

max_epochs     = 300
best_val_dice  = 0.0
best_epoch     = 0
train_loss_log = []
val_dice_log   = []
start_epoch    = 1
ACCUM_STEPS    = 4
PATIENCE       = 50
no_improve     = 0


if RESUME_FROM and os.path.exists(RESUME_FROM):
    checkpoint = torch.load(RESUME_FROM, map_location=device)

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        best_val_dice  = checkpoint["best_val_dice"]
        train_loss_log = checkpoint.get("train_loss_log", [])
        val_dice_log   = checkpoint.get("val_dice_log", [])
        start_epoch    = checkpoint["epoch"] + 1
        print(f"Resumed full checkpoint from epoch {checkpoint['epoch']}")
        print(f"Best Dice so far: {best_val_dice:.4f}")
    else:
        model.load_state_dict(checkpoint)
        print(f"Loaded model weights from {RESUME_FROM}")
        print("Optimizer and scheduler reset to initial state.")
        print("Note: train_loss_log and val_dice_log start fresh.")
else:
    print("Starting training from scratch.")


for epoch in range(start_epoch, max_epochs + 1):

    
    model.train()
    epoch_loss  = 0.0
    epoch_steps = 0

    with tqdm(train_loader, desc=f"Epoch {epoch:>3}/{max_epochs} [Train]", leave=False) as pbar:
        for step, batch in enumerate(pbar):

            if isinstance(batch["image"], list):
                images_b = torch.cat(batch["image"], dim=0).to(device)
                labels_b = torch.cat(batch["label"], dim=0).to(device)
            else:
                images_b = batch["image"].to(device)
                labels_b = batch["label"].to(device)

            outputs = model(images_b)
            loss    = loss_fn(outputs, labels_b)

           
            loss = loss / ACCUM_STEPS
            loss.backward()

            
            if (step + 1) % ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
                torch.cuda.empty_cache()

            epoch_loss  += loss.item() * ACCUM_STEPS   
            epoch_steps += 1
            pbar.set_postfix(loss=f"{loss.item() * ACCUM_STEPS:.4f}")

       
        optimizer.step()
        optimizer.zero_grad()
        torch.cuda.empty_cache()

    avg_train_loss = epoch_loss / epoch_steps
    train_loss_log.append(avg_train_loss)

    
    if epoch % 5 == 0:
        model.eval()
        torch.cuda.empty_cache()

        with torch.no_grad():
            with tqdm(val_loader, desc=f"Epoch {epoch:>3}/{max_epochs} [Val]  ", leave=False) as pbar:
                for val_batch in pbar:

                    if isinstance(val_batch["image"], list):
                        val_images = torch.cat(val_batch["image"], dim=0).to(device)
                        val_labels = torch.cat(val_batch["label"], dim=0).to(device)
                    else:
                        val_images = val_batch["image"].to(device)
                        val_labels = val_batch["label"].to(device)

                    
                    val_outputs = inferer(val_images, model)

                    val_outputs_list = decollate_batch(val_outputs)
                    val_labels_list  = decollate_batch(val_labels)
                    val_outputs_post = [post_pred(x)  for x in val_outputs_list]
                    val_labels_post  = [post_label(x) for x in val_labels_list]

                    dice_metric(y_pred=val_outputs_post, y=val_labels_post)

                    
                    del val_images, val_labels, val_outputs
                    torch.cuda.empty_cache()

        mean_dice = dice_metric.aggregate().item()
        dice_metric.reset()
        val_dice_log.append((epoch, mean_dice))

       
        scheduler.step(mean_dice)

        print(f"Epoch [{epoch:>3}/{max_epochs}]  "
              f"Train Loss: {avg_train_loss:.4f}  |  "
              f"Val Dice: {mean_dice:.4f}")

        
        if mean_dice > best_val_dice:
            best_val_dice = mean_dice
            best_epoch    = epoch
            no_improve    = 0
            torch.save(model.state_dict(), "best_swin_unetr.pth")
            print(f"  ✓ New best model saved (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1
            print(f"  No improvement for {no_improve} val checks "
                  f"(patience: {PATIENCE})")
            if no_improve >= PATIENCE:
                print(f"\n⚠ Early stopping triggered at epoch {epoch}")
                print(f"Best Val Dice: {best_val_dice:.4f} at epoch {best_epoch}")
                break

    else:
        print(f"Epoch [{epoch:>3}/{max_epochs}]  Train Loss: {avg_train_loss:.4f}")

    
    if epoch % 10 == 0:
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_dice":        best_val_dice,
            "train_loss_log":       train_loss_log,
            "val_dice_log":         val_dice_log,
        }, f"checkpoint_epoch_{epoch}.pth")
        print(f"  💾 Checkpoint saved at epoch {epoch}")

    
    torch.cuda.empty_cache()

print(f"\nTraining complete.")
print(f"Best Val Dice: {best_val_dice:.4f} at epoch {best_epoch}")

In [ ]:
import matplotlib.pyplot as plt

epochs_logged = list(range(1, len(train_loss_log) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


axes[0].plot(epochs_logged, train_loss_log, color="steelblue", linewidth=1)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("DiceCE Loss")
axes[0].grid(True, alpha=0.3)


if val_dice_log:
    val_epochs = [x[0] for x in val_dice_log]
    val_dices  = [x[1] for x in val_dice_log]
    axes[1].plot(val_epochs, val_dices, color="darkorange", 
                 linewidth=1, marker="o", markersize=3)
    axes[1].set_title("Validation Dice Score")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

print(f"Total epochs run  : {len(train_loss_log)}")
print(f"Best train loss   : {min(train_loss_log):.4f} at epoch {train_loss_log.index(min(train_loss_log)) + 1}")
if val_dice_log:
    best_val = max(val_dice_log, key=lambda x: x[1])
    worst_val = min(val_dice_log, key=lambda x: x[1])
    print(f"Best  val Dice    : {best_val[1]:.4f} at epoch {best_val[0]}")
    print(f"Worst val Dice    : {worst_val[1]:.4f} at epoch {worst_val[0]}")
    print(f"Latest val Dice   : {val_dice_log[-1][1]:.4f}")